# Entity Linking & Disambiguation

NER found "Paris". Entity linking decides: 
Parise, Frane ? Partis Hilton? Paris, Texas...

## Problem Definition

Entity linking resolves each mention to a unique entry in a knowledge base: Wikadata, Wikipedia .. Two subtasks:
1. Candidate generation.
    Give "Jordan", which KB entries are plausible.
2. Disambiguation.
    Given the context, which candidate is the right one?

## Basic Concept

Entity linking pipeline: mention -> candidates -> disambiguated entity

### Candidate generation

Given the mention surface from "Jordan", look up candidates in an alias index. Typical index returns 10~30 candidates per mention.

### Disambiguation: three approaches

1. Prior + context.

    P(entity | mention) x context_similarity(entity, text)

2. Embedding-based.

    Encode mention + context. Encode each candidate's description, Pick max cosine.

3. Generative.

    Decode the entity's canonical name token by token. Constrained to a trie of valid entity names so output is guaranteed to be a valid KB id.

### End-to-end vs pipline

run NER + candidate generation + disambiguation in one pass

### Measurements

1. Metion recall.
    Fraction of gold mentions where **the correct KB entry appears in the candidate list.**

2. Disambiguation accuracy.
    Given correct candidates, **how often the top-1 is right.**

# Buid your Own

## Build an alias index from Wikipedia redirects

In [2]:
alias_to_entities = {
    "jordan": ["Q41421 (Michael Jordan)", "Q810 (Jordan, country)", "Q254110 (Michael B. Jordan)"],
    "paris":  ["Q90 (Paris, France)", "Q663094 (Paris, Texas)", "Q55411 (Paris Hilton)"],
    "apple":  ["Q312 (Apple Inc.)", "Q89 (apple, fruit)"],
}

## Context-based disambiguation

In [6]:
def disambiguate(mention, context, alias_index, entity_desc):
    candidates = alias_index.get(mention.lower(), [])
    if not candidates:
        return None, 0.0
    context_words = set(tokenize(context))
    best, best_score = None, 0.0
    for entity_id in candidates:
        key = entity_id.split(" (")[0]
        desc_words = set(tokenize(entity_desc[key]))
        union = len(context_words | desc_words)
        score = len(context_words & desc_words) / union if union else 0.0
        if score > best_score:
            best, best_score = entity_id, score
    return best, best_score

def tokenize(text):
    return text.lower().split()

ENTITY_DESC = {
    "Q41421": "Michael Jordan",
    "Q810": "Jordan, country",
    "Q254110": "Michael B. Jordan",
    "Q90": "Paris, France",
    "Q663094": "Paris, Texas",
    "Q55411": "Paris Hilton",
    "Q312": "Apple Inc.",
    "Q89": "apple, fruit",
}

import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

with SectionPrinter("Context-based disambiguation"):
    print(disambiguate("paris", "I'm going to Paris France", alias_to_entities, ENTITY_DESC))


================Context-based disambiguation================
('Q90 (Paris, France)', 0.16666666666666666)


## Embedding-based

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_mention(text, mention_span):
    start, end = mention_span
    marked = f"{text[:start]} [MENTION] {text[start:end]} [/MENTION] {text[end:]}"
    return encoder.encode([marked], normalize_embeddings=True)[0]

def embed_entity(entity_id, description):
    return encoder.encode([f"{entity_id}: {description}"], normalize_embeddings=True)[0]

with SectionPrinter("Embedding-based disambiguation"):
    text = "I'm going to Paris France"
    mention_span = (13, 18)
    candidates = alias_to_entities["paris"]

    mention_emb = embed_mention(text, mention_span)
    entity_embs = [
        embed_entity(candidate.split(" (")[0], ENTITY_DESC[candidate.split(" (")[0]])
        for candidate in candidates
    ]
    sims = np.array([mention_emb @ emb for emb in entity_embs])

    best_idx = int(sims.argmax())
    print(candidates[best_idx], sims[best_idx])
    

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

===============Embedding-based disambiguation===============
Q90 (Paris, France) 0.55514234


# Pitfalls

* NIL handling.   Some mentions are not in KB, systems must perdict NIL instead of guessing the wrong entity.
* Mention Bounary Errors.    Upstream NER misses partial spans ("Bank of America" tagged as just "Bank")
* Popularity bias.  Trained systems over-predict frequent entities.
* Cross-lingual EL.   Mapping mentions in Chinese text to English Wikipedia entities.
* KB staleness.  New companies, events, people are not in last year's Wikipedia dump. Production pipelines need a refresh loop.